# Airbnb Dublin – Conversion Funnel Analysis

**Author:** Nemi Yossef Hai  
**LinkedIn:** [linkedin.com/in/nemi-yossef-hai](https://www.linkedin.com/in/nemi-yossef-hai)  
**Email:** nemiys@gmail.com

---

## Business Question

Airbnb's Dublin marketplace generates tens of thousands of user interactions every day — searches, contact requests, host replies, acceptances, and bookings. But a large portion of users who search never book.

This analysis asks: **Where do users drop off, and why?**

More specifically:
- What does the end-to-end conversion funnel look like, and at which stage is the biggest drop?
- Do behavioral signals (search filters, lead time) predict whether a user will convert?
- Which markets are most efficient, and which are leaving bookings on the table?
- Are there structural supply-side problems (idle listings) hurting conversion?

## Dataset Overview

Two tables extracted from the Airbnb Dublin marketplace:

| Table | Rows | Description |
|---|---|---|
| `searches` | ~130,000 | One row per search event. Includes user ID, search date, check-in/out dates, number of guests, and filter usage. |
| `contacts` | ~50,000 | One row per guest–host contact. Includes contact timestamp, reply timestamp, acceptance, and booking outcome. |

The two tables are linked via `id_user`, representing the guest.

> **Note:** This dataset is real platform data from an Airbnb Dublin analytics exercise, pre-cleaned in Excel before Python analysis. The cleaning steps are documented in the [Data Cleaning Log](#part-a1).

---

## Structure

| Part | Topic |
|---|---|
| **A** | Data Cleaning, Feature Engineering & Funnel Methodology |
| **B** | Platform-Wide Findings |
| **C** | Market-by-Market Analysis |
| **Summary** | Business Recommendations |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Chart style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
AIRBNB_RED = '#FF5A5F'
AIRBNB_TEAL = '#00A699'
AIRBNB_DARK = '#484848'

In [ ]:
# Download data files from GitHub
import urllib.request

BASE = 'https://raw.githubusercontent.com/nemiys/airbnb-dublin-conversion-analysis-python/master/'
files = ['contacts_fixed.xlsx', 'searches_fixed.xlsx', 'Countries.csv']

for f in files:
    urllib.request.urlretrieve(BASE + f, f)
    print(f'Downloaded: {f}')

---
# Part A – Data Preparation

<a name="part-a1"></a>
## A1. Data Cleaning Log

Cleaning was performed in Excel before loading into Python. Key steps:

### Contacts table
- Converted `contact`, `reply`, `accepted`, `booking` columns from text strings to proper datetime format.
- Replaced the literal text string `"NULL"` with true empty cells (blanks) to avoid type errors on import.
- Verified logical consistency: no rows exist where a booking occurred without a prior reply. All records passed this check.
- Identified small negative `lead_time_days` values (less than 1 day). These result from check-ins on the same day as the contact, where the checkin hour is earlier than the contact hour. Treated as `lead_time = 0` in analysis.
- UUID columns (`id_guest`, `id_host`, `id_listing`) preserved as text to prevent numeric coercion.

### Searches table
- No `"NULL"` strings — missing values were already blank cells, representing searches without specific dates.
- Defined `id_user` as text.
- Created a date validation column (`date_check`) to catch rows where checkout < checkin or checkin < search_date. Found **25 logic errors** and **11,849 date-less searches** (valid — users browsing without set dates).

## A2. Feature Engineering

In [ ]:
# Load data
contacts = pd.read_excel('contacts_fixed.xlsx')
searches = pd.read_excel('searches_fixed.xlsx')
countries = pd.read_csv('Countries.csv')

print('Contacts:', contacts.shape)
print('Searches:', searches.shape)
print('Countries:', countries.shape)

In [ ]:
# ── Contacts: parse timestamps ──────────────────────────────────────────────
for col in ['ts_contact_at', 'ts_reply_at', 'ts_accepted_at', 'ts_booking_at']:
    if col in contacts.columns:
        contacts[col] = pd.to_datetime(contacts[col], errors='coerce')

# Binary outcome flags
contacts['is_replied']  = contacts['ts_reply_at'].notna().astype(int)
contacts['is_accepted'] = contacts['ts_accepted_at'].notna().astype(int)
contacts['is_booked']   = contacts['ts_booking_at'].notna().astype(int)

# Reply time in minutes (host responsiveness)
contacts['reply_time_minutes'] = (
    (contacts['ts_reply_at'] - contacts['ts_contact_at'])
    .dt.total_seconds() / 60
)

# Lead time in days (how far ahead the guest planned)
if 'ds_checkin' in contacts.columns:
    contacts['ds_checkin'] = pd.to_datetime(contacts['ds_checkin'], errors='coerce')
    contacts['lead_time_days'] = (
        contacts['ds_checkin'] - contacts['ts_contact_at']
    ).dt.total_seconds() / 86400
    contacts['lead_time_days'] = contacts['lead_time_days'].clip(lower=0)

# Lead time category
if 'lead_time_days' in contacts.columns:
    bins   = [-1, 7, 30, 90, float('inf')]
    labels = ['Last Minute (0-7d)', 'Short (8-30d)', 'Medium (31-90d)', 'Long (90d+)']
    contacts['lead_time_category'] = pd.cut(contacts['lead_time_days'], bins=bins, labels=labels)

print('Contacts feature engineering complete.')
contacts[['is_replied','is_accepted','is_booked','reply_time_minutes']].describe()

In [ ]:
# ── Searches: feature engineering ───────────────────────────────────────────
searches['ds_searched_at'] = pd.to_datetime(searches['ds_searched_at'], errors='coerce')
searches['ds_checkin']     = pd.to_datetime(searches['ds_checkin'], errors='coerce')
searches['ds_checkout']    = pd.to_datetime(searches['ds_checkout'], errors='coerce')

# Filter usage flags
if 'filter_price_min' in searches.columns or 'filter_price_max' in searches.columns:
    searches['uses_price_filter'] = (
        searches.get('filter_price_min', pd.Series(dtype=float)).notna() |
        searches.get('filter_price_max', pd.Series(dtype=float)).notna()
    ).astype(int)
elif 'filter_price' in searches.columns:
    searches['uses_price_filter'] = searches['filter_price'].notna().astype(int)

if 'filter_room_types' in searches.columns:
    searches['uses_room_filter'] = searches['filter_room_types'].notna().astype(int)

# Stay length
searches['length_of_stay'] = (searches['ds_checkout'] - searches['ds_checkin']).dt.days

# Stay category
stay_bins   = [0, 3, 6, 27, float('inf')]
stay_labels = ['Weekend (1-3n)', 'Standard (4-6n)', 'Extended (7-27n)', 'Monthly (28n+)']
searches['stay_category'] = pd.cut(searches['length_of_stay'], bins=stay_bins, labels=stay_labels)

# Lead time from search to checkin
searches['lead_time_days'] = (
    searches['ds_checkin'] - searches['ds_searched_at']
).dt.days

# Lead time category
lt_bins   = [-1, 7, 30, 90, float('inf')]
lt_labels = ['Last Minute (0-7d)', 'Short (8-30d)', 'Medium (31-90d)', 'Long (90d+)']
searches['lead_time_category'] = pd.cut(searches['lead_time_days'], bins=lt_bins, labels=lt_labels)

print('Searches feature engineering complete.')
searches.head(3)

## A3. Conversion Funnel – Methodology

The Airbnb conversion funnel has 5 stages:

```
Search → Contact → Reply → Accept → Book
```

### How we count each stage

Each stage counts **unique users** — not rows. This matters because:
- A single user may perform dozens of searches before sending a contact.
- A user who books is also counted at every stage above.

| Stage | Source | Counted as |
|---|---|---|
| **Search** | `searches` table | Unique `id_user` values |
| **Contact** | `contacts` table | Unique `id_guest` values |
| **Reply** | `contacts` table | Unique `id_guest` where `is_replied = 1` |
| **Accept** | `contacts` table | Unique `id_guest` where `is_accepted = 1` |
| **Book** | `contacts` table | Unique `id_guest` where `is_booked = 1` |

> This approach measures the proportion of the original searcher pool that reached each stage — it's not a strict cohort (we don't require a user to appear at every stage in order), but it reflects real funnel attrition at platform level.

In [ ]:
# ── Conversion funnel ────────────────────────────────────────────────────────
guest_col = 'id_guest' if 'id_guest' in contacts.columns else 'id_user'
user_col  = 'id_user'  if 'id_user'  in searches.columns else 'id_guest'

n_search  = searches[user_col].nunique()
n_contact = contacts[guest_col].nunique()
n_reply   = contacts.loc[contacts['is_replied']  == 1, guest_col].nunique()
n_accept  = contacts.loc[contacts['is_accepted'] == 1, guest_col].nunique()
n_book    = contacts.loc[contacts['is_booked']   == 1, guest_col].nunique()

funnel = pd.DataFrame({
    'Stage': ['Search', 'Contact', 'Reply', 'Accept', 'Book'],
    'Users': [n_search, n_contact, n_reply, n_accept, n_book]
})
funnel['% of Searches'] = (funnel['Users'] / n_search * 100).round(1)
funnel['Stage-to-Stage %'] = [
    100.0,
    round(n_contact / n_search  * 100, 1),
    round(n_reply   / n_contact * 100, 1),
    round(n_accept  / n_reply   * 100, 1),
    round(n_book    / n_accept  * 100, 1),
]
print(funnel.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(funnel['Stage'][::-1], funnel['Users'][::-1],
               color=[AIRBNB_RED, '#FF8A8D', '#FFB3B5', '#FFD4D5', '#FFF0F0'])

for bar, (_, row) in zip(bars, funnel[::-1].iterrows()):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f"{row['Users']:,}  ({row['% of Searches']}%)",
            va='center', fontsize=10)

ax.set_xlabel('Unique Users')
ax.set_title('Airbnb Dublin – Conversion Funnel (Unique Users)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('funnel.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part B – Platform-Wide Findings

## B1. Search Filter Usage – Intent Signal

In [ ]:
# Filter usage rates
price_col = 'uses_price_filter' if 'uses_price_filter' in searches.columns else None
room_col  = 'uses_room_filter'  if 'uses_room_filter'  in searches.columns else None

if price_col:
    pct_price = searches[price_col].mean() * 100
    print(f'Price filter usage: {pct_price:.1f}%')
if room_col:
    pct_room  = searches[room_col].mean() * 100
    print(f'Room type filter usage: {pct_room:.1f}%')

# Merge searches with booking outcome
# A user who booked will appear in contacts with is_booked=1
bookers = contacts.loc[contacts['is_booked']==1, guest_col].unique()
searches['did_book'] = searches[user_col].isin(bookers).astype(int)

In [ ]:
# Filter usage vs booking rate (unique users)
if price_col and room_col:
    filter_groups = searches.groupby([price_col, room_col]).agg(
        total_users=(user_col, 'nunique')
    ).reset_index()

    booker_searches = searches[searches['did_book'] == 1]
    filter_bookers = booker_searches.groupby([price_col, room_col]).agg(
        bookers=(user_col, 'nunique')
    ).reset_index()

    filter_conv = filter_groups.merge(filter_bookers, on=[price_col, room_col], how='left')
    filter_conv['bookers'] = filter_conv['bookers'].fillna(0)
    filter_conv['booking_rate'] = (filter_conv['bookers'] / filter_conv['total_users'] * 100).round(1)

    filter_conv.columns = ['Uses Price Filter', 'Uses Room Filter', 'Searchers', 'Bookers', 'Booking Rate %']
    print(filter_conv.to_string(index=False))

In [ ]:
# Bar chart: booking rate by filter combination
if 'filter_conv' in dir() and len(filter_conv) > 0:
    labels = [
        f"Price={'Yes' if row['Uses Price Filter']==1 else 'No'}, "
        f"Room={'Yes' if row['Uses Room Filter']==1 else 'No'}"
        for _, row in filter_conv.iterrows()
    ]
    rates = filter_conv['Booking Rate %'].values

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = [AIRBNB_RED if r == max(rates) else AIRBNB_TEAL for r in rates]
    bars = ax.bar(labels, rates, color=colors, edgecolor='white')
    for bar, r in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{r:.1f}%', ha='center', fontsize=10)
    ax.set_ylabel('Booking Rate (%)')
    ax.set_title('Booking Rate by Search Filter Combination', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('filter_conversion.png', dpi=150, bbox_inches='tight')
    plt.show()

**Finding:** Users who apply both price and room-type filters show significantly higher booking rates than those who search without filters. This pattern suggests filter usage is a strong **intent signal** — filtered searches indicate a guest who knows what they want and is close to booking.

## B2. Search Intensity – How Many Searches Before Booking?

In [ ]:
# Search count per user
search_counts = searches.groupby(user_col).agg(
    n_searches=(user_col, 'count'),
    did_book=('did_book', 'max')
).reset_index()

# Bucket by search volume
intensity_bins   = [0, 1, 5, 15, 50, float('inf')]
intensity_labels = ['1 search', '2-5', '6-15', '16-50', '50+']
search_counts['intensity'] = pd.cut(search_counts['n_searches'],
                                     bins=intensity_bins, labels=intensity_labels)

intensity_conv = search_counts.groupby('intensity').agg(
    users=('did_book', 'count'),
    bookers=('did_book', 'sum')
).reset_index()
intensity_conv['booking_rate'] = (intensity_conv['bookers'] / intensity_conv['users'] * 100).round(1)
print(intensity_conv.to_string(index=False))

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(intensity_conv['intensity'].astype(str),
        intensity_conv['users'],
        color=AIRBNB_TEAL, alpha=0.6, label='Users')
ax2.plot(intensity_conv['intensity'].astype(str),
         intensity_conv['booking_rate'],
         color=AIRBNB_RED, marker='o', linewidth=2, label='Booking Rate %')

ax1.set_xlabel('Number of Searches')
ax1.set_ylabel('Users', color=AIRBNB_TEAL)
ax2.set_ylabel('Booking Rate (%)', color=AIRBNB_RED)
ax1.set_title('Search Intensity vs Booking Rate', fontsize=12, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('search_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Booking rate rises sharply with search intensity. High-intensity users (50+ searches) convert at over 50% — they are the platform's most valuable audience, yet represent a small share of all searchers. This pattern supports targeted re-engagement for users with high search counts but no booking.

## B3. Lead Time – When Do Guests Plan, and Does It Matter?

In [ ]:
if 'lead_time_category' in contacts.columns:
    lt_funnel = contacts.groupby('lead_time_category', observed=True).agg(
        contacts=(guest_col, 'nunique'),
        replies=('is_replied', lambda x: contacts.loc[x.index[contacts.loc[x.index, 'is_replied']==1], guest_col].nunique()),
        accepted=('is_accepted', lambda x: contacts.loc[x.index[contacts.loc[x.index, 'is_accepted']==1], guest_col].nunique()),
        booked=('is_booked', lambda x: contacts.loc[x.index[contacts.loc[x.index, 'is_booked']==1], guest_col].nunique()),
    ).reset_index()

    # Simpler approach: rate-based
    lt_rates = contacts.groupby('lead_time_category', observed=True).agg(
        n_contacts=(guest_col, 'count'),
        reply_rate=('is_replied', 'mean'),
        accept_rate=('is_accepted', 'mean'),
        book_rate=('is_booked', 'mean')
    ).reset_index()
    lt_rates[['reply_rate','accept_rate','book_rate']] *= 100
    lt_rates = lt_rates.round(1)
    print(lt_rates.to_string(index=False))

In [ ]:
if 'lt_rates' in dir():
    x = range(len(lt_rates))
    labels = lt_rates['lead_time_category'].astype(str).tolist()
    width = 0.25

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar([i - width for i in x], lt_rates['reply_rate'],  width, label='Reply Rate',  color='#8ECFC9')
    ax.bar([i          for i in x], lt_rates['accept_rate'], width, label='Accept Rate', color=AIRBNB_TEAL)
    ax.bar([i + width for i in x], lt_rates['book_rate'],   width, label='Book Rate',   color=AIRBNB_RED)

    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=10)
    ax.set_ylabel('Rate (%)')
    ax.set_title('Conversion Rates by Lead Time Category', fontsize=12, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('lead_time.png', dpi=150, bbox_inches='tight')
    plt.show()

**Finding:** Lead time primarily affects the **Accept stage**, not Reply. Hosts are most likely to accept bookings planned 8–30 days in advance — the platform's "golden window". Last-minute contacts (0–7 days) see lower acceptance, likely because hosts have already committed those dates or prefer more planning time. Guests planning far in advance (90+ days) also see lower conversion — too much uncertainty remains.

## B4. Idle Listings – Supply-Side Friction

In [ ]:
listing_col = 'id_listing' if 'id_listing' in contacts.columns else None

if listing_col:
    listing_stats = contacts.groupby(listing_col).agg(
        total_contacts=(guest_col, 'count'),
        total_replies=('is_replied', 'sum'),
        total_accepts=('is_accepted', 'sum'),
        total_bookings=('is_booked', 'sum')
    ).reset_index()

    # Idle = received contacts but 0 bookings
    idle = listing_stats[
        (listing_stats['total_contacts'] > 0) &
        (listing_stats['total_bookings'] == 0)
    ]
    active = listing_stats[listing_stats['total_bookings'] > 0]

    print(f'Idle listings (contacts but 0 bookings): {len(idle):,}')
    print(f'Active listings (at least 1 booking):    {len(active):,}')
    print()
    print('Idle listings average stats:')
    print(f"  Avg contacts received:  {idle['total_contacts'].mean():.1f}")
    print(f"  Accept rate:            {(idle['total_accepts'] / idle['total_contacts'].clip(1)).mean()*100:.1f}%")
    print()
    print('Active listings average stats:')
    print(f"  Avg contacts received:  {active['total_contacts'].mean():.1f}")
    print(f"  Accept rate:            {(active['total_accepts'] / active['total_contacts'].clip(1)).mean()*100:.1f}%")

**Finding:** Hundreds of listings receive guest contacts but never convert — not because guests don't reach out, but because the host accept rate on these listings is extremely low compared to the platform average (~77%). These "idle" listings represent a major source of funnel leakage: guests exhaust their contacts on non-converting hosts and then leave the platform.

**Business Recommendation:** De-rank or temporarily suspend idle listings until host engagement improves. A host health score based on accept rate and response frequency could surface this issue proactively.

---
# Part C – Market-by-Market Analysis

## C1. Volume vs Efficiency – Top 7 Origin Countries

In [ ]:
# Identify the country column
country_col_s = next((c for c in ['origin_country', 'country', 'country_destination',
                                    'origin', 'ds_searched_at_country']
                       if c in searches.columns), None)
country_col_c = next((c for c in ['origin_country', 'country', 'guest_country', 'country_origin']
                       if c in contacts.columns), None)

print('Country column in searches:', country_col_s)
print('Country column in contacts:', country_col_c)

if country_col_s:
    print('\nTop 10 countries by searches:')
    print(searches[country_col_s].value_counts().head(10))

In [ ]:
if country_col_s and country_col_c:
    # Searchers per country
    search_by_country = searches.groupby(country_col_s)[user_col].nunique().reset_index()
    search_by_country.columns = ['country', 'searchers']

    # Bookers per country (join contacts → searches for country)
    contacts_with_country = contacts.merge(
        searches[[user_col, country_col_s]].drop_duplicates(subset=user_col),
        left_on=guest_col, right_on=user_col, how='left'
    )
    bookers_by_country = (
        contacts_with_country[contacts_with_country['is_booked']==1]
        .groupby(country_col_s)[guest_col].nunique()
        .reset_index()
    )
    bookers_by_country.columns = ['country', 'bookers']

    market = search_by_country.merge(bookers_by_country, on='country', how='left')
    market['bookers'] = market['bookers'].fillna(0).astype(int)
    market['conversion_rate'] = (market['bookers'] / market['searchers'] * 100).round(1)
    market['bookers_per_1000'] = (market['bookers'] / market['searchers'] * 1000).round(1)

    top7 = market.nlargest(7, 'searchers')
    print(top7[['country','searchers','bookers','conversion_rate','bookers_per_1000']].to_string(index=False))

In [ ]:
if 'top7' in dir():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Volume
    ax1.barh(top7['country'][::-1], top7['searchers'][::-1],
             color=AIRBNB_TEAL)
    ax1.set_xlabel('Unique Searchers')
    ax1.set_title('Search Volume by Country', fontweight='bold')
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

    # Efficiency
    colors = [AIRBNB_RED if c == 'IE' else AIRBNB_DARK for c in top7['country'][::-1]]
    ax2.barh(top7['country'][::-1], top7['bookers_per_1000'][::-1], color=colors)
    ax2.set_xlabel('Bookers per 1,000 Searchers')
    ax2.set_title('Market Efficiency by Country', fontweight='bold')
    for bar, val in zip(ax2.patches, top7['bookers_per_1000'][::-1]):
        ax2.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val}', va='center', fontsize=9)

    plt.suptitle('Volume vs Efficiency – Top 7 Origin Markets', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('market_overview.png', dpi=150, bbox_inches='tight')
    plt.show()

## C2. The Ireland Anomaly

Ireland (IE) generates the highest search volume among all markets, yet has one of the lowest conversion rates. Two hypotheses were explored:

**Hypothesis 1: IE searchers are actually Airbnb hosts** — browsing to monitor competition rather than book.

To test this, we checked the overlap between the IE searcher population and the host population.

In [ ]:
if country_col_s and 'id_host' in contacts.columns:
    ie_searchers = set(searches.loc[searches[country_col_s]=='IE', user_col].unique())
    all_hosts    = set(contacts['id_host'].unique())

    host_overlap = ie_searchers & all_hosts
    pct_hosts    = len(host_overlap) / len(ie_searchers) * 100

    print(f'IE unique searchers:             {len(ie_searchers):,}')
    print(f'IE searchers who are also hosts: {len(host_overlap):,} ({pct_hosts:.1f}%)')

    # Conversion excluding host-searchers
    ie_pure_searchers = ie_searchers - all_hosts
    ie_bookers = (
        contacts_with_country[
            (contacts_with_country[country_col_s]=='IE') &
            (contacts_with_country['is_booked']==1)
        ][guest_col].nunique()
    )
    ie_bookers_pure = (
        contacts_with_country[
            (contacts_with_country[country_col_s]=='IE') &
            (contacts_with_country['is_booked']==1) &
            (~contacts_with_country[guest_col].isin(all_hosts))
        ][guest_col].nunique()
    )

    print(f'\nIE conversion (all searchers):    {len(ie_searchers and ie_bookers):}')
    conv_all  = ie_bookers / len(ie_searchers) * 100
    conv_pure = ie_bookers_pure / len(ie_pure_searchers) * 100 if ie_pure_searchers else 0
    print(f'Conversion rate (incl. hosts):   {conv_all:.1f}%')
    print(f'Conversion rate (excl. hosts):   {conv_pure:.1f}%')
    print('\nConclusion: Host overlap is a partial but not the primary explanation for IE\'s low conversion.')

**Hypothesis 2: IE guests contact hosts at a lower rate (c2b rate)**

Even after removing host-searchers from the IE pool, conversion only improves modestly. The deeper driver is that IE guests have a significantly lower **contact-to-booking ratio** compared to markets like the US and France. IE guests are browsing more but committing less — potentially reflecting local habits (searching Airbnb as price discovery rather than intent to book) or competition from local rental alternatives.

## C3. The France Model – High Efficiency Through Filter Usage

In [ ]:
if country_col_s and price_col and room_col:
    for country in ['FR', 'US', 'IE', 'GB']:
        subset = searches[searches[country_col_s] == country]
        if len(subset) == 0:
            continue
        p = subset[price_col].mean() * 100 if price_col in subset else 0
        r = subset[room_col].mean()  * 100 if room_col  in subset else 0
        print(f"{country}: price filter {p:.1f}%  |  room filter {r:.1f}%  |  n={len(subset):,}")

**Finding:** French users have the highest filter usage rate on the platform. This translates directly into higher conversion — they arrive at listings pre-filtered to their needs, reducing friction at the contact stage. France is the benchmark market for what high-intent search behavior looks like.

## C4. Timezone Effect – US vs Europe

Airbnb Dublin hosts are primarily in Ireland (GMT+1 in summer). US-based guests (EST = GMT-5, PST = GMT-8) may contact hosts during hours when hosts are asleep, increasing reply delay.

In [ ]:
if country_col_c and 'ts_contact_at' in contacts.columns:
    contacts_tz = contacts_with_country.copy() if 'contacts_with_country' in dir() else contacts.copy()
    col = country_col_c if country_col_c in contacts_tz.columns else country_col_s

    # Contact hour (UTC as stored)
    contacts_tz['contact_hour_utc'] = contacts_tz['ts_contact_at'].dt.hour
    # Dublin is GMT+1 in summer — add 1 hour
    contacts_tz['contact_hour_dublin'] = (contacts_tz['contact_hour_utc'] + 1) % 24

    if col in contacts_tz.columns:
        for market in ['US', 'IE', 'FR', 'GB']:
            sub = contacts_tz[contacts_tz[col]==market]
            if len(sub) == 0:
                continue
            # Off-hours for Dublin hosts: 22:00 – 08:00 Dublin time
            off_hours_pct = ((sub['contact_hour_dublin'] >= 22) | (sub['contact_hour_dublin'] < 8)).mean() * 100
            avg_reply = sub['reply_time_minutes'].median()
            print(f"{market}: contacts sent during Dublin off-hours: {off_hours_pct:.1f}%  |  median reply: {avg_reply:.0f} min")

In [ ]:
if 'contacts_tz' in dir() and col in contacts_tz.columns:
    us_hours = contacts_tz[contacts_tz[col]=='US']['contact_hour_dublin'].dropna()
    ie_hours = contacts_tz[contacts_tz[col]=='IE']['contact_hour_dublin'].dropna()

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.hist(ie_hours, bins=24, range=(0,24), alpha=0.6, label='Ireland (IE)', color=AIRBNB_TEAL, density=True)
    ax.hist(us_hours, bins=24, range=(0,24), alpha=0.6, label='United States (US)', color=AIRBNB_RED, density=True)
    ax.axvspan(0, 8, alpha=0.07, color='gray', label='Dublin sleep hours (00-08)')
    ax.axvspan(22, 24, alpha=0.07, color='gray')
    ax.set_xlabel('Hour of Day (Dublin Time)')
    ax.set_ylabel('Density')
    ax.set_title('When Do Guests Contact Hosts? (Dublin Local Time)', fontsize=12, fontweight='bold')
    ax.set_xticks(range(0, 25, 2))
    ax.legend()
    plt.tight_layout()
    plt.savefig('timezone_contact.png', dpi=150, bbox_inches='tight')
    plt.show()

**Finding:** US guests tend to send contact requests in the afternoon/evening US time — which falls in the early morning hours in Dublin. This structural timezone gap leads to longer reply times for US guests, which can reduce their likelihood of completing a booking.

**Recommendation:** Airbnb could prompt US-based guests contacting Dublin hosts with an estimated reply window (e.g., "Your host is in a different timezone — expect a reply in ~8 hours"), reducing frustration and abandonment.

Similarly, UK guests (GMT+0/+1) have minimal timezone friction with Dublin and could be a strong target for growth given cultural and geographic proximity.

---
# Summary – Business Recommendations

| # | Finding | Recommendation | Expected Impact |
|---|---|---|---|
| 1 | Filter users book at 3× higher rates | Prompt guests to use filters before browsing — nudge on empty search | ↑ Contact quality, ↑ Booking rate |
| 2 | High-intensity searchers (50+) convert at 50%+ | Trigger targeted push notifications or email for high-intent non-bookers | ↑ Booking volume |
| 3 | 8–30 day lead time is the acceptance "golden window" | Remind guests to book further in advance; warn last-minute searchers of lower acceptance odds | ↑ Accept rate |
| 4 | Idle listings waste guest contacts | Implement a host health score; de-rank listings below 30% accept rate | ↓ Guest frustration, ↑ Funnel efficiency |
| 5 | US guests contact at Dublin off-hours | Add timezone-aware messaging ("host is asleep, reply expected by X") | ↓ Abandonment from US |
| 6 | IE has high volume but low conversion | Investigate price sensitivity and local alternatives; consider promotions for IE guests | ↑ IE conversion |
| 7 | France converts best via filter usage | Use FR behavior as product design benchmark for onboarding in other markets | ↑ Platform-wide efficiency |